# Ea results description

## GA for tuning 

### Ex 1

**Model**
```python
class NeuralNetwork(nn.Module): # define a neural network class
    def __init__(self): # initialisation method
        super().__init__() # initialise the neural network
        self.flatten = nn.Flatten() # flatten (appiattire) the input image
        self.linear_relu_stack = nn.Sequential( # create a sequence of layers
            nn.Linear(28*28, 512), # linear layer from 28*28 to 512 (28*28 is the size of the input image, 512 is the size of the output tensor)
            nn.ReLU(),  # activation function
            nn.Linear(512, 512), # linear layer from 512 to 512 (512 is the size of the input tensor, 512 is the size of the output tensor)
            nn.ReLU(), # activation function
            nn.Linear(512, 10) # linear layer from 512 to 10 (512 is the size of the input tensor, 10 is the size of the output tensor)
        )

    def forward(self, x): # forward method
        x = self.flatten(x) # flatten the input image
        logits = self.linear_relu_stack(x) # pass the input image through the sequence of layers
        return logits # return the output tensor
```

**EA parameters**
```python
best_lr_individual = ea_for_lr(      
        max_fitness_evaluations=50,
        train_dataloader=train_dataloader,
        validation_dataloader=validation_dataloader,
        population_size = 10,
        num_epochs_per_evaluation = 5, # change to 5
        mutation_probability = 0.1,
        solution_upper_bound = 1e-2,
        solution_lower_bound = 1e-4,
        num_roulette_selected = 3)
```

**LR**

Naive lr:
10^-3

Tuned lr:
0.009592190579976487

**Plots**

are called ea1_train, ea1_test, ea1_test_acc

### Ex 2

I built a NN class in which you can specify all the paramters that you want, then i compare the same model with different paramters.

```python

    def __init__(self, 
                 input_size = 28*28,
                 hidden_sizes = [512,512],
                 output_size = 10,
                 activation_function = nn.ReLU,
                 initialize_weights = False): # initialisation method
        super().__init__() # initialise the neural network
        self.flatten = nn.Flatten() # flatten (appiattire) the input image
        self.activation_function = activation_function()

        # Create a list to hold the layers
        layers = []
        previous_size = input_size

        # Add hidden layers dynamically
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(previous_size, hidden_size))
            layers.append(self.activation_function)
            previous_size = hidden_size
        layers.append(nn.Linear(previous_size, output_size)) # Add the final output layer

        # Create a sequential container with the layers
        # You shoould pass a list of layers to the Sequential container, one by one. 
        # You can do this by using the * operator to unpack the list of layers
        # This is done to avoid of passing a list of layers to the Sequential container (that is not correct)
        self.linear_relu_stack = nn.Sequential(*layers)
      
        if initialize_weights:
            self._initialize_weights()

    def forward(self, x): # forward method
        x = self.flatten(x) # flatten the input image
        logits = self.linear_relu_stack(x) # pass the input image through the sequence of layers
        return logits # return the output tensor
    
    def _initialize_weights(self):
      for m in self.modules():
          if isinstance(m, nn.Linear):
              nn.init.xavier_uniform_(m.weight)
              nn.init.zeros_(m.bias)
```

The results are in plots:
- changePar_1_loss (changing simple par)
- changePar_1_acc (changing simple par)
- changePar_2_loss (changing layers and units)
- changePar_2_acc (changing layers and units)

### Ex 3
A ES to tune the activation function 

```python
# Ea for finding the best actiavtion function 

def ga_for_af(
        max_fitness_evaluations, # 50 is suggested 
        population_size=5,
        activation_functions_gene_space=[0, 1, 2, 3, 4],
        train_dataloader=train_dataloader,
        validation_dataloader=validation_dataloader,
        num_epochs_per_evaluation=5,
        lr = 0.0096,
        input_size = 28*28,
        hidden_sizes = [512,512],
        output_size = 10,
        initialize_weights = False,
        num_roulette_selected=3,
        mutation_probability=0.1
        ):
    
    # Initialize the population
    fitness_evaluations = 0
    gen_index = 1
    population = initialization(population_size, activation_functions_gene_space)
    fitness_evaluations += population_size
    evaluate_population(
        population,
        train_dataloader,
        validation_dataloader,
        num_epochs_per_evaluation=num_epochs_per_evaluation,
        lr = lr,
        input_size = input_size,
        hidden_sizes = hidden_sizes,
        output_size = output_size,
        initialize_weights = initialize_weights)
    best_individual = find_best_individual(population)
    print(f"Generation: {gen_index} | Fitness evalutions: {fitness_evaluations} | Best individual: {best_individual}")

    while fitness_evaluations < max_fitness_evaluations:
        # Select parents using roulette wheel selection
        parents = roulette_wheel_selection(population, best_individual, num_roulette_selected)
        # Add the best individual to the parents
        parents.append(best_individual)
        # Create a new population through crossover and mutation
        population = create_new_population(parents, population_size, mutation_probability)
        population.append(best_individual)
        # Evaluate the new population
        evaluate_population(
            population,
            train_dataloader,
            validation_dataloader,
            num_epochs_per_evaluation=num_epochs_per_evaluation,
            lr = lr,
            input_size = input_size,
            hidden_sizes = hidden_sizes,
            output_size = output_size,
            initialize_weights = initialize_weights)
        fitness_evaluations += population_size

        # Find the best individual in the new population
        best_individual = find_best_individual(population)
        print(f"Generation: {gen_index + 1} | Fitness evalutions: {fitness_evaluations} | Best individual: {best_individual}")
        gen_index += 1

    return best_individual
```

The result plots are in:
- ea for af training loss
- ea for af accuracy 

### Ex 4  
An ES to tune the number and size of layer (where the genotype space is composed by classes)

```python
# Ea for finding the best actiavtion function 

def ga_for_size_and_number_of_layers(
        max_fitness_evaluations, # 50 is suggested 
        crossover_operators,
        mutation_operators,
        population_size=5,
        train_dataloader=train_dataloader,
        validation_dataloader=validation_dataloader,
        num_epochs_per_evaluation=5,
        lr = 0.0096,
        number_of_layers_interval=[1, 5],
        number_of_units_space=[16, 32, 64, 128, 256, 512],
        input_size = 28*28,
        output_size = 10,
        activation_function = nn.LeakyReLU,
        initialize_weights = False,
        num_roulette_selected=3
        ):
    
    # Initialize the population
    fitness_evaluations = 0
    gen_index = 1
    population = initialization(population_size, number_of_layers_interval, number_of_units_space)
    fitness_evaluations += population_size
    evaluate_population(
        population,
        train_dataloader,
        validation_dataloader,
        num_epochs_per_evaluation=num_epochs_per_evaluation,
        lr = lr,
        input_size = input_size,
        output_size = output_size,
        activation_function=activation_function,
        initialize_weights = initialize_weights)
    best_individual = find_best_individual(population)
    print(f"\nGeneration: {gen_index} | Fitness evalutions: {fitness_evaluations} | Best individual: {best_individual}")
    for individual in population:
        print(individual)

    while fitness_evaluations < max_fitness_evaluations:
        # Select parents using roulette wheel selection
        parents = roulette_wheel_selection(population, best_individual, num_roulette_selected)
        # Add the best individual to the parents
        best_individual_copy = Individual(gene=best_individual.gene.copy()) # to avoid mutation of the best individual
        parents.append(best_individual_copy)
        # Create a new population through crossover and mutation
        population = create_new_population(parents, population_size, mutation_operators, crossover_operators, number_of_units_space)
        population.append(best_individual)
        # Evaluate the new population
        evaluate_population(
            population,
            train_dataloader,
            validation_dataloader,
            num_epochs_per_evaluation=num_epochs_per_evaluation,
            lr = lr,
            input_size = input_size,
            output_size = output_size,
            activation_function=activation_function,
            initialize_weights = initialize_weights)
        fitness_evaluations += population_size

        # Find the best individual in the new population
        best_individual = find_best_individual(population)
        print(f"\nGeneration: {gen_index + 1} | Fitness evalutions: {fitness_evaluations} | Best individual: {best_individual}")
        for individual in population:
            print(individual)
        gen_index += 1

    return best_individual
```

Results in:
- es integer for NNStructure testing loss
- es integer for NNStructure accuracy

### Ex 5
An ES to tune the number and size of layer (where the genotype space is an interval in the set of the natural numbers)

```python
# Ea for finding the best actiavtion function 

def es_lambda_plus_mu_for_size_and_units(
        max_fitness_evaluations, # 50 is suggested 
        beginning_population_size,
        offsprings_size,
        number_of_layers_interval,
        number_of_units_interval,
        percentage_of_population_size_for_tournament,
        crossover_operator,
        integer_mutation_probability_per_layer,
        percentage_of_integer_mutation,
        grow_shrink_mutation_operator_check,
        grow_shrink_mutation_probability,
        minimum_gene,
        maximum_gene,
        train_dataloader=train_dataloader,
        validation_dataloader=validation_dataloader,
        num_epochs_per_evaluation=5, # 10 is suggested 
        lr = 0.0096,
        input_size = 28*28,
        output_size = 10,
        activation_function = nn.LeakyReLU,
        initialize_weights = False
        ):
    
    # Initialize the population
    fitness_evaluations = 0
    gen_index = 1
    population = initialization(
        population_size = beginning_population_size,
        number_of_layers_interval = number_of_layers_interval,
        number_of_units_interval = number_of_units_interval)
    fitness_evaluations += beginning_population_size
    evaluate_population(
        population = population,
        training_data = train_dataloader,
        validation_data = validation_dataloader,
        num_epochs_per_evaluation=num_epochs_per_evaluation,
        lr = lr,
        input_size = input_size,
        output_size = output_size,
        activation_function = activation_function,
        initialize_weights = initialize_weights)
       
    best_individual = find_best_individual(population)
    # Print the status of the population
    print(f"\nGeneration: {gen_index} | Fitness evalutions: {fitness_evaluations} | Best individual: {best_individual}")
    for individual in population:
        print(individual)

    while fitness_evaluations < max_fitness_evaluations:
        # Select parents using tournament selection
        parents = tournament_selection(
            population = population,
            number_of_selected=offsprings_size, 
            percentage_of_population_size=percentage_of_population_size_for_tournament)
        # Create offsprings
        offsprings = create_offsprings(
            parents = parents,
            offsprings_size = offsprings_size,
            crossover_operator=crossover_operator,
            integer_mutation_probability_per_layer=integer_mutation_probability_per_layer,
            percentage_of_integer_mutation = percentage_of_integer_mutation,
            grow_shrink_mutation_operator_check=grow_shrink_mutation_operator_check,
            grow_shrink_mutation_probability=grow_shrink_mutation_probability,
            minimum_gene=minimum_gene,
            maximum_gene=maximum_gene)
        for offspring in offsprings:
            print(offspring)
        # Evaluate the new population
        evaluate_population(
            population = offsprings,
            training_data = train_dataloader,
            validation_data = validation_dataloader,
            num_epochs_per_evaluation=num_epochs_per_evaluation,
            lr = lr,
            input_size = input_size,
            output_size = output_size,
            activation_function=activation_function,
            initialize_weights = initialize_weights)
        for offspring in offsprings:
            population.append(offspring) # Add the offsprings to the population, one by one :)  
        fitness_evaluations += offsprings_size

        # Find the best individual in the new population
        best_individual = find_best_individual(population)
        print(f"\nGeneration: {gen_index + 1} | Fitness evalutions: {fitness_evaluations} | Best individual: {best_individual}")
        for individual in population:
            print(individual)
        gen_index += 1

    return best_individual
```

Result (for the case of image recognition): 

Standard architecture: [512,512]

Best individual: Gene: [369, 584.0, 263.5, 429.25, 233.5], Solution: [369, 584, 264, 429, 234], Fitness: 0.37009229590284065

Results in:
- es real for NNStructure loss
- es real for NNStructure accuracy

Comment:
The population size wass small and the maximum number of fitness eavluation was small. 
Moreover, the shrink and grow operator was blocked in order to focus only on the arithmetic crossover and mutation.
Furthermore, the lr was chosen as the best lr for the standard configuration, while the evolutionary best structure was chosen as the best configuration, once fixed the lr rate as the best for the standard configuration.
With these considerations the evolutionary strategy was depowered.
However, the results were better then the one of the standard structure. 
This is a good news, because with precautions on the paramters of the ES and with more computational power the ES can become even better.

## Ex 6
Regarding learning the nn structure, i tried v3 vs v4 and the results are in 

- v3_vs_v4

Attention: \
maybe i run the tuning of the both with different paramters, so i am doing a second rn in order to have a better comparison. 

- v3_vs_v4_2